# EP02 — Transformer Self-Attention
**COMPSCI 713 · S1 2024 Q2 · 11 marks**

| Part | Topic | Marks |
|------|-------|-------|
| a | Name the three vectors in self-attention | 6 |
| b | Advantage of self-attention over traditional attention | 5 |

---

## Part a — Three Vectors in Self-Attention [6 marks]

For each input token vector **x_i**, three vectors are created using learned weight matrices:

| Vector | Created by | Role |
|--------|-----------|------|
| **Query (q)** | x_i * W_Q | 'What am I looking for?' — used to score against all keys |
| **Key (k)** | x_i * W_K | 'What do I represent?' — scored against every query |
| **Value (v)** | x_i * W_V | 'What do I contribute?' — weighted and summed for output |

**The computation:**
```
score(q_i, k_j) = dot(q_i, k_j) / sqrt(d_k)   <- scaled dot-product
weights_i       = softmax(scores_i)              <- attention distribution
output_i        = sum_j( weights_ij * v_j )      <- weighted value sum
```

Each output position is a mixture of all value vectors, weighted by how much each key matched the query at that position.

---

## Part b — Advantage of Self-Attention [5 marks]

**Traditional (cross) attention** (e.g., Bahdanau in seq2seq):
- Query comes from the *decoder* hidden state
- Keys/Values come from *encoder* outputs
- Only captures alignment between encoder and decoder
- Computed sequentially — limited by RNN's sequential bottleneck

**Self-attention** (same sequence attends to itself):

| Advantage | Details |
|-----------|--------|
| **Direct long-range dependencies** | Any token can attend to any other token in O(1) steps — no vanishing gradient over distance |
| **Fully parallelisable** | All positions computed simultaneously — no sequential recurrence like RNNs/LSTMs |
| **Bidirectional context** | Each token sees the entire sequence at once (encoders); no directional bottleneck |
| **Richer representations** | Multiple heads capture different types of relationships simultaneously |
| **Interpretability** | Attention weights show which tokens influence each position |

---

## Code Demo — Scaled Dot-Product Self-Attention from scratch

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt

class SelfAttention(nn.Module):
    'Single-head scaled dot-product self-attention.'
    def __init__(self, d_model):
        super().__init__()
        self.scale = d_model ** 0.5
        self.W_Q = nn.Linear(d_model, d_model, bias=False)  # Query
        self.W_K = nn.Linear(d_model, d_model, bias=False)  # Key
        self.W_V = nn.Linear(d_model, d_model, bias=False)  # Value

    def forward(self, x):
        Q = self.W_Q(x)                                   # (B, T, d)
        K = self.W_K(x)                                   # (B, T, d)
        V = self.W_V(x)                                   # (B, T, d)
        scores  = torch.bmm(Q, K.transpose(1,2)) / self.scale  # (B, T, T)
        weights = F.softmax(scores, dim=-1)                     # (B, T, T)
        out     = torch.bmm(weights, V)                         # (B, T, d)
        return out, weights


torch.manual_seed(42)
d_model, seq_len = 16, 6
x = torch.randn(1, seq_len, d_model)

attn = SelfAttention(d_model)
out, W = attn(x)
print('Input :', x.shape, '  Output:', out.shape)
print('Attention weight matrix shape:', W.shape, '  (each row sums to 1)')
print('Row 0 sum:', W[0,0].sum().item())

fig, ax = plt.subplots(figsize=(5,4))
im = ax.imshow(W[0].detach(), cmap='Blues', vmin=0, vmax=W[0].max().item())
ax.set_xlabel('Key position (j)'); ax.set_ylabel('Query position (i)')
ax.set_title('Self-Attention Matrix\nRow i: how much token i attends to each token j')
plt.colorbar(im, ax=ax); plt.tight_layout(); plt.show()